In [23]:
# Setup: Import required libraries
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

True

In [27]:
# Initialize OpenAI client
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODERATION_MODEL = os.getenv("MODERATION_MODEL")
client = OpenAI(api_key=OPENAI_API_KEY)

print(f"✓ API Key: {'Found' if OPENAI_API_KEY else 'MISSING'}")
print(f"✓ Moderation Model: {MODERATION_MODEL}")

✓ API Key: Found
✓ Moderation Model: omni-moderation-latest


In [32]:
# Simple moderation function
def is_safe_content(text: str) -> bool:
    """Check if content is safe using OpenAI Moderation API"""
    response = client.moderations.create(input=text)
    result = response.results[0]
    
    # Return False if any category is flagged
    return not result.flagged

In [33]:
# Test with safe content
safe_text = "What is the weather in New York?"

if is_safe_content(safe_text):
    print(f"✓ Safe: '{safe_text}'")
else:
    print(f"✗ Blocked: '{safe_text}'")

✓ Safe: 'What is the weather in New York?'


In [ ]:
# Test with unsafe content
unsafe_text = "I want to hurt someone"

if is_safe_content(unsafe_text):
    print(f"✓ Safe: '{unsafe_text}'")
else:
    print(f"✗ Blocked: '{unsafe_text}'")

✗ Blocked: 'I want to hurt someone'


In [37]:
# Detailed moderation with category breakdown
def moderate_with_details(text: str):
    response = client.moderations.create(input=text)
    result = response.results[0]
    
    print(f"Input: '{text}'")
    print(f"Flagged: {result.flagged}")
    print(f"\nCategories:")
    print(f"  Sexual: {result.categories.sexual}")
    print(f"  Hate: {result.categories.hate}")
    print(f"  Violence: {result.categories.violence}")
    print(f"  Self-harm: {result.categories.self_harm}")
    
    return result

## Production Flow: Moderation-First Pattern

```
User Input
    ↓
┌─────────────────────┐
│ Moderation Check    │ ← Check FIRST (before LLM)
│ (OpenAI Mod API)    │
└─────────────────────┘
    ↓
  Safe?
    ├─ No  → Block & Return Error Message
    │
    ├─ Yes → ┌──────────────┐
    │        │  Agent/LLM   │
    │        └──────────────┘
    │              ↓
    │        ┌──────────────────────┐
    │        │ Optional: Check      │
    │        │ Output Moderation    │
    │        └──────────────────────┘
    │              ↓
    └────────→ Return Response
```

**Why moderation first?**
- Saves cost (don't call LLM for unsafe inputs)
- Faster rejection
- Prevents prompt injection
- Cleaner logs

In [42]:
# Production pattern: Safe chat with input + output moderation
from langchain_core.messages import HumanMessage

def safe_chat(user_input: str, agent, config):
    """
    Safe chat wrapper with input and output moderation.
    Returns response only if both input and output are safe.
    """
    # Step 1: Input guardrail - check BEFORE agent
    try:
        if not is_safe_content(user_input):
            return {
                "status": "blocked",
                "reason": "input_violation",
                "message": "I cannot process this request due to content policy."
            }
    except Exception as e:
        return {
            "status": "error",
            "reason": "moderation_failed",
            "message": f"Input moderation error: {str(e)}"
        }
    
    # Step 2: Invoke agent only if input is safe
    response = agent.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config
    )
    
    # Step 3: Output guardrail - check AI response
    ai_message = response['messages'][-1].content
    try:
        if not is_safe_content(ai_message):
            return {
                "status": "blocked",
                "reason": "output_violation",
                "message": "Generated content was blocked by safety filters."
            }
    except Exception as e:
        # If output moderation fails, still return the response with a warning
        return {
            "status": "warning",
            "reason": "output_moderation_failed",
            "message": ai_message,
            "warning": f"Could not verify output safety: {str(e)}",
            "full_response": response
        }
    
    # Step 4: Return safe response
    return {
        "status": "success",
        "message": ai_message,
        "full_response": response
    }

print("✓ safe_chat function ready for production use (with error handling)")

✓ safe_chat function ready for production use (with error handling)


In [40]:
model

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15', 'langchain-openai': '1.4.3'}}, profile={'name': 'GPT-5.4 nano', 'release_date': '2026-03-17', 'last_updated': '2026-03-17', 'open_weights': False, 'max_input_tokens': 400000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh']}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000174FA2DE4E0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0

In [43]:
# Example: Test safe_chat (requires agent from previous notebooks)
# Uncomment when you have an agent configured:

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
 
model = init_chat_model(os.getenv("CHAT_MODEL"))
agent = create_agent(model=model, checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "safe-test"}}

# Test safe input
result = safe_chat("What is 2+2?", agent, config)
print(result)
 
# Test unsafe input
result = safe_chat("I want to hurt someone", agent, config)
print(result)

print("Safe chat example (commented out - uncomment to test with your agent)")

{'status': 'success', 'message': '2 + 2 = **4**.', 'full_response': {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='fbbecb60-681c-4a5e-b65d-becdb2340e37'), AIMessage(content='2 + 2 = **4**.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 13, 'total_tokens': 25, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EBpSLw9CVbhkTMmMPh1ZWr6QazvJZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ff2fd-4403-7a01-b033-f0ca15965c59-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 12, 'total_tokens': 25, '